# 11 · Reranking

**Idea:** a first-stage retriever favors **recall** — return many candidates cheaply. A **reranker** then favors **precision**, scoring each candidate against the query with a more expensive model and reordering. Here the default reranker is an **LLM-as-judge** (rate each passage 0–10). We also track its cost.

> Runs on Ollama by default. One cell optionally uses OpenAI — it skips cleanly if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import load_markdown, NumpyStore

corpus = load_markdown()
store = NumpyStore(provider='ollama'); store.add(corpus)

query = 'why does chunk size matter for retrieval quality?'
wide = store.search(query, k=8)   # stage 1: recall
print('stage-1 candidates:')
for h in wide:
    print(f'  {h.score:5.2f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')

Now rerank those 8 candidates down to the best 3 with an LLM judge, recording cost as we go.

In [ ]:
from rag_lab import Reranker
from study_buddy import CostTracker

tracker = CostTracker()
reranker = Reranker(provider='ollama', tracker=tracker)
top = reranker.rerank(query, wide, k=3)
print('reranked top-3 (LLM relevance score):')
for h in top:
    print(f'  {h.score:5.2f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')
print('\ncost:', tracker.summary())

The score column is now a **relevance judgement**, not a cosine distance — reranking can promote a chunk the vector store ranked lower. That precision costs one LLM call per candidate, which is why we rerank a *small* stage-1 set.

Optionally, rerank with a hosted model (OpenAI). Skips if no key is set.

In [ ]:
try:
    cloud = CostTracker()
    top_cloud = Reranker(provider='openai', model='gpt-4o-mini', tracker=cloud).rerank(query, wide, k=3)
    for h in top_cloud:
        print(f'  {h.score:5.2f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')
    print('cost:', cloud.summary())
except Exception as e:
    print('openai skipped:', type(e).__name__, e)

> A classic **cross-encoder** reranker is also available via `cross_encoder_rerank` if you `pip install sentence-transformers`. It reads query+passage together for a sharp score without an LLM call.

**Takeaway:** retrieve wide, rerank narrow. A reranker buys precision at the cost of a few model calls — budget it by keeping the stage-1 candidate set small.

Next → `12_context_compression.ipynb`